In [ ]:
from pathlib import Path
import json
import subprocess
import sys

def run_cmd(cmd):
    print(' '.join(cmd))
    proc = subprocess.run(cmd, text=True, capture_output=True)
    if proc.stdout:
        print(proc.stdout)
    if proc.returncode != 0:
        if proc.stderr:
            print(proc.stderr)
        raise RuntimeError(f'Command failed with code {proc.returncode}')
    return proc

def read_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

In [ ]:
PYTHON = sys.executable  # s3po-gs env
SCRIPT = Path('/home/bzhang512/CV_Project/part3_BRPO/scripts/prepare_stage1_difix_dataset_s3po.py')

# === Config ===
SCENE_NAME = '405841'
RUN_KEY = '405841__s3po__midpoint__v1'
DATASET_ROOT = Path('/home/bzhang512/CV_Project/dataset')
STORAGE_ROOT = Path('/home/bzhang512/my_storage_500G/CV_Project')

# === S3PO external eval output ===
EXTERNAL_EVAL_RUN = '2026-04-04-00-43-20'  # from SLAM run timestamp
EXTERNAL_EVAL_DIR = STORAGE_ROOT / 'output/part2_s3po/00_external_eval' / SCENE_NAME / f'infer_{SCENE_NAME}_full' / EXTERNAL_EVAL_RUN

# === Input paths ===
SPARSE_MANIFEST = DATASET_ROOT / SCENE_NAME / 'part2_s3po/sparse/split_manifest.json'
TEST_MANIFEST = DATASET_ROOT / SCENE_NAME / 'part2_s3po/test/split_manifest.json'
SPARSE_RGB_DIR = DATASET_ROOT / SCENE_NAME / 'part2_s3po/sparse/rgb'
RENDER_RGB_DIR = EXTERNAL_EVAL_DIR / 'render_rgb'
RENDER_DEPTH_DIR = EXTERNAL_EVAL_DIR / 'render_depth_npy'
TRJ_JSON = EXTERNAL_EVAL_DIR / 'plot/trj_external_infer.json'

# === Difix params ===
PLACEMENT = 'midpoint'
PROMPT = 'remove degradation'
DIFIX_MODEL_NAME = 'nvidia/difix_ref'
HEIGHT = 512
WIDTH = 512
TIMESTEP = 199

print('RUN_ROOT =', DATASET_ROOT / SCENE_NAME / 'part3_stage1' / RUN_KEY)
print('EXTERNAL_EVAL_DIR =', EXTERNAL_EVAL_DIR)
print('RENDER_RGB_DIR =', RENDER_RGB_DIR)
print('TRJ_JSON =', TRJ_JSON)

In [ ]:
# Check inputs exist
assert SPARSE_MANIFEST.exists(), f'Missing {SPARSE_MANIFEST}'
assert TEST_MANIFEST.exists(), f'Missing {TEST_MANIFEST}'
assert SPARSE_RGB_DIR.exists(), f'Missing {SPARSE_RGB_DIR}'
assert RENDER_RGB_DIR.exists(), f'Missing {RENDER_RGB_DIR}'
assert RENDER_DEPTH_DIR.exists(), f'Missing {RENDER_DEPTH_DIR}'
assert TRJ_JSON.exists(), f'Missing {TRJ_JSON}'
print('All inputs exist')

# Show sparse/test info
sparse_manifest = read_json(SPARSE_MANIFEST)
test_manifest = read_json(TEST_MANIFEST)
print(f"Sparse: {len(sparse_manifest['selected_indices'])} frames")
print(f"Test: {len(test_manifest['selected_indices'])} frames")
print(f"Sparse indices: {sparse_manifest['selected_indices']}")

In [ ]:
# SELECT
cmd = [
    PYTHON, str(SCRIPT),
    '--stage', 'select',
    '--scene-name', SCENE_NAME,
    '--run-key', RUN_KEY,
    '--dataset-root', str(DATASET_ROOT),
    '--sparse-manifest', str(SPARSE_MANIFEST),
    '--test-manifest', str(TEST_MANIFEST),
    '--sparse-rgb-dir', str(SPARSE_RGB_DIR),
    '--render-rgb-dir', str(RENDER_RGB_DIR),
    '--render-depth-dir', str(RENDER_DEPTH_DIR),
    '--trj-json', str(TRJ_JSON),
    '--placement', PLACEMENT,
]
run_cmd(cmd)

RUN_ROOT = DATASET_ROOT / SCENE_NAME / 'part3_stage1' / RUN_KEY
summary = read_json(RUN_ROOT / 'manifests' / 'selection_summary.json')
summary

In [ ]:
# DIFIX
cmd = [
    PYTHON, str(SCRIPT),
    '--stage', 'difix',
    '--scene-name', SCENE_NAME,
    '--run-key', RUN_KEY,
    '--dataset-root', str(DATASET_ROOT),
    '--sparse-manifest', str(SPARSE_MANIFEST),
    '--test-manifest', str(TEST_MANIFEST),
    '--sparse-rgb-dir', str(SPARSE_RGB_DIR),
    '--render-rgb-dir', str(RENDER_RGB_DIR),
    '--render-depth-dir', str(RENDER_DEPTH_DIR),
    '--trj-json', str(TRJ_JSON),
    '--difix-model-name', DIFIX_MODEL_NAME,
    '--prompt', PROMPT,
    '--height', str(HEIGHT),
    '--width', str(WIDTH),
    '--timestep', str(TIMESTEP),
]
run_cmd(cmd)

difix_manifest = read_json(RUN_ROOT / 'manifests' / 'difix_run_manifest.json')
difix_manifest

In [ ]:
# PACK
cmd = [
    PYTHON, str(SCRIPT),
    '--stage', 'pack',
    '--scene-name', SCENE_NAME,
    '--run-key', RUN_KEY,
    '--dataset-root', str(DATASET_ROOT),
    '--sparse-manifest', str(SPARSE_MANIFEST),
    '--test-manifest', str(TEST_MANIFEST),
    '--sparse-rgb-dir', str(SPARSE_RGB_DIR),
    '--render-rgb-dir', str(RENDER_RGB_DIR),
    '--render-depth-dir', str(RENDER_DEPTH_DIR),
    '--trj-json', str(TRJ_JSON),
]
run_cmd(cmd)

pack_manifest = read_json(RUN_ROOT / 'manifests' / 'pack_manifest.json')
pack_manifest

In [ ]:
# QUICK CHECK
left_dir = RUN_ROOT / 'difix' / 'left_fixed'
right_dir = RUN_ROOT / 'difix' / 'right_fixed'
aug_left = RUN_ROOT / 'augmented_train_left' / 'rgb'
aug_right = RUN_ROOT / 'augmented_train_right' / 'rgb'
pseudo_cache = RUN_ROOT / 'pseudo_cache'

print('left_fixed_count  =', len(list(left_dir.glob('*.png'))))
print('right_fixed_count =', len(list(right_dir.glob('*.png'))))
print('aug_left_count    =', len(list(aug_left.glob('*.png'))))
print('aug_right_count   =', len(list(aug_right.glob('*.png'))))
print('pseudo_cache samples =', len(list((pseudo_cache / 'samples').glob('*'))))
print('sample left  =', sorted(p.name for p in left_dir.glob('*.png'))[:5])
print('sample right =', sorted(p.name for p in right_dir.glob('*.png'))[:5])

In [ ]:
# Check pseudo_cache structure
cache_manifest = read_json(pseudo_cache / 'manifest.json')
print('num_samples:', cache_manifest['num_samples'])
print('sample_ids:', cache_manifest['sample_ids'])

# Check first sample
first_sample = pseudo_cache / 'samples' / str(cache_manifest['sample_ids'][0])
print('First sample files:', [p.name for p in first_sample.glob('*')])

# Check camera.json
camera = read_json(first_sample / 'camera.json')
print('camera keys:', list(camera.keys()))
print('frame_id:', camera['frame_id'])
print('test_idx:', camera['test_idx'])